In [ ]:
!pip install -U ragas langchain-ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [ ]:
import os
import time
import torch
import pandas as pd

from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel

from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.metrics.collections import (
    faithfulness,
    answer_correctness,
    summarization_score
)
from langchain_ollama import ChatOllama, OllamaEmbeddings

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_11564/1286948830.py:11: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_11564/1286948830.py:11: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics im

### Set up Ollama

In [ ]:
!apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!nohup /usr/local/bin/ollama serve > ollama_server.log 2>&1 &

time.sleep(10)
print("Ollama server started")

!/usr/local/bin/ollama list

Ollama server started. Check ollama_server.log for details.
NAME                       ID              SIZE      MODIFIED       
llama3.1:8b                46e0c10c039e    4.9 GB    23 minutes ago    
nomic-embed-text:latest    0a109f422b47    274 MB    2 hours ago       
qwen2.5:1.5b               65ec06548149    986 MB    2 hours ago       


### Pull models for ragas

In [ ]:
#!/usr/local/bin/ollama pull qwen2.5:1.5b
!/usr/local/bin/ollama pull llama3.1:8b
!/usr/local/bin/ollama pull nomic-embed-text

## Configuration

In [ ]:
MODEL_PATH = "distilled-qwen-student"
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
USE_BASELINE = True

DATASET_NAME = "hynky/czech_news_dataset_v2"
SPLIT = "train[:100]"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

GENERATED_SUMMARIES_PATH = "generated_summaries.json"
EVALUATION_RESULTS_PATH = "evaluation_results.csv"

RAGAS_HF_EMBEDDING_MODEL = "sentence-transformers/distiluse-base-multilingual-cased"

## Helper Functions

## Sumarization helper

In [ ]:
# Model loading
def get_model_dtype():
    return torch.float16 if DEVICE == "cuda" else torch.float32


def load_model_and_tokenizer(model_path, use_baseline=False):
    model_dtype = get_model_dtype()

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)

    if use_baseline:
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME,
            torch_dtype=model_dtype,
            device_map="auto",
        )
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME,
            torch_dtype=model_dtype,
            device_map="auto",
        )
        model = PeftModel.from_pretrained(base_model, model_path)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.eval()
    return model, tokenizer

# Prompting
PROMPT_PREFIX = (
    "Jsi profesionální český novinář a editor. "
    "Tvým úkolem je vytvořit výstižné, přesné a neutrální shrnutí dodaného "
    "novinového článku POUZE V ČEŠTINĚ. Shrnutí by mělo obsahovat nejdůležitější fakta a NESMÍ "
    "si vymýšlet žádné nové informace, domněnky ani názory. Buď stručný a věcný.\n\n"
    "Přečti si následující článek a napiš jeho stručné shrnutí "
    "(maximálně 2 až 3 věty).\n\n"
    "ČLÁNEK:\n"
)


def generate_summary(model, tokenizer, article):
    messages = [
        {
            "role": "system",
            "content": (
                "Jsi profesionální český novinář a editor. "
                "Tvým úkolem je vytvořit výstižné, přesné a neutrální shrnutí "
                "dodaného novinového článku POUZE V ČEŠTINĚ. Shrnutí by mělo obsahovat "
                "nejdůležitější fakta a NESMÍ si vymýšlet žádné nové informace, "
                "domněnky ani názory. Buď stručný a věcný."
            ),
        },
        {
            "role": "user",
            "content": (
                "Přečti si následující článek a napiš jeho stručné shrnutí "
                "(maximálně 2 až 3 věty).\n\n"
                f"ČLÁNEK:\n{article}"
            ),
        },
    ]

    device = next(model.parameters()).device

    if hasattr(tokenizer, "apply_chat_template"):
        model_inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )

        model_inputs = {k: v.to(device) for k, v in model_inputs.items()}
        prompt_length = model_inputs["input_ids"].shape[1]

        with torch.no_grad():
            output_ids = model.generate(
                **model_inputs,
                max_new_tokens=120,
                do_sample=False,
                repetition_penalty=1.1,
                no_repeat_ngram_size=3,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )

        generated_ids = output_ids[0][prompt_length:]
        return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # Fallback for tokenizers without chat template support
    prompt = f"{PROMPT_PREFIX}{article}\n\nSHRNUTÍ:"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    prompt_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_ids = output_ids[0][prompt_length:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def prepare_evaluation_data(dataset, model, tokenizer):
    eval_data = []

    start = time.time()
    total = len(dataset)

    for idx, item in enumerate(dataset, 1):
        article = item["content"]
        ground_truth = item.get("brief", "")
        contexts = [article]

        start_gen = time.time()
        summary = generate_summary(model, tokenizer, article)
        elapsed_gen = time.time() - start_gen

        print("Summary:", summary)
        print("Ground truth:", ground_truth)

        eval_data.append(
            {
                "user_input": "Shrň tento článek do stručného shrnutí.",
                "retrieved_contexts": contexts,
                "response": summary,
                "reference": ground_truth,
                "reference_contexts": contexts,
            }
        )

        if idx % 5 == 0 or idx == total:
            print(
                f"Prepared {idx}/{total} examples "
                f"(last gen {elapsed_gen:.2f}s, elapsed {(time.time() - start):.2f}s)"
            )

    total_time = time.time() - start
    print(f"Finished preparing {total} examples in {total_time:.2f}s")
    return eval_data

# Save/Load generated Summaries
def save_generated_summaries(eval_data, output_path=GENERATED_SUMMARIES_PATH):
    df = pd.DataFrame(eval_data)
    df.to_json(
        output_path,
        orient="records",
        indent=4,
        force_ascii=False,
    )
    print(f"Generated summaries saved to {output_path}")


def load_generated_summaries(input_path=GENERATED_SUMMARIES_PATH):
    df = pd.read_json(input_path)
    records = df.to_dict(orient="records")
    print(f"Loaded {len(records)} records from {input_path}")
    return records


## Ragas evaluation

In [ ]:
def run_evaluation(eval_data):
    eval_dataset = Dataset.from_pandas(pd.DataFrame(eval_data))

    ragas_llm = ChatOllama(
        model="qwen2.5:1.5b",
        base_url="http://localhost:11434",
        temperature=0.0,
    )

    ragas_embeddings = OllamaEmbeddings(
        model="nomic-embed-text",
        base_url="http://localhost:11434",
    )

    run_config = RunConfig(
        max_workers=1,
        max_retries=3,
        timeout=120,
    )

    metrics = [
        #faithfulness,
        answer_correctness,
        #summarization_score,
    ]

    results = evaluate(
        eval_dataset,
        metrics=metrics,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        run_config=run_config,
        raise_exceptions=False,
    )

    print("Results:")
    print(results)
    return results


def save_evaluation_results(results, output_path=EVALUATION_RESULTS_PATH):
    results_df = results.to_pandas().drop(
        columns=["user_input", "retrieved_contexts"],
        errors="ignore",
    )
    results_df = results_df.rename(
        columns={
            "reference_contexts": "Reference contexts",
            "reference": "Reference summary",
            "response": "Generated summary",
            "answer_correctness": "Answer Correctness",
        }
    )
    results_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(f"Evaluation results saved to {output_path}")

## Summarization Pipeline


In [ ]:
def run_summarization_stage():
    print("Summarization started")
    t_start = time.time()

    print("Loading dataset...")
    t = time.time()
    dataset = load_dataset(DATASET_NAME, split=SPLIT)
    print(f"Dataset loaded ({len(dataset)} examples) in {(time.time() - t):.2f}s")

    print("Loading model and tokenizer...")
    t = time.time()
    model, tokenizer = load_model_and_tokenizer(
        MODEL_PATH,
        use_baseline=USE_BASELINE,
    )
    print(f"Loaded model in {(time.time() - t):.2f}s")

    print("Generating summaries...")
    t = time.time()
    eval_data = prepare_evaluation_data(dataset, model, tokenizer)
    print(f"Generated summaries in {(time.time() - t):.2f}s")

    save_generated_summaries(eval_data, GENERATED_SUMMARIES_PATH)

    print(f"Summarization completed in {(time.time() - t_start):.2f}s ---")

## Evaluation Pipeline

In [ ]:
def run_evaluation_stage():
    print("Evaluation started")
    t_start = time.time()

    print("Loading generated summaries...")
    t = time.time()
    eval_data = load_generated_summaries(GENERATED_SUMMARIES_PATH)
    print(f"Loaded summaries in {(time.time() - t):.2f}s")

    print("Running RAGAS evaluation...")
    t = time.time()
    results = run_evaluation(eval_data)
    print(f"Evaluation done in {(time.time() - t):.2f}s")

    save_evaluation_results(results, EVALUATION_RESULTS_PATH)

    print(f"Evaluation completed in {(time.time() - t_start):.2f}s")

## Run Summarization

In [ ]:

run_summarization_stage()

--- Evaluation stage started ---
[1/2] Loading generated summaries...
Loaded 100 records from generated_summaries.json
Loaded summaries in 0.02s
[2/2] Running RAGAS evaluation...


Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt statement_generator_prompt failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.executor:Exception raised in Job[98]: RagasOutputParserException(The output parser failed to parse the output including retries.)


Results:
{'answer_correctness': 0.4423}
Evaluation done in 1251.35s
Evaluation results saved to evaluation_results.csv
--- Evaluation stage completed in 1251.44s ---


## Run Evaluation

In [ ]:
run_evaluation_stage()